# Notebook 01 — Crowd Density Estimation

**Goal:** Train and evaluate two architectures on ShanghaiTech (A+B) and JHU-Crowd++:
1. **CSRNet** (CVPR 2018 baseline)
2. **AdaptiveCSRNet** (ours: CBAM attention + multi-scale ASPP backend)

**Evaluation metrics:** MAE, MSE, PSNR, SSIM, GAME(0–3)

> All cells run automatically. Training state is checkpointed — interrupted runs resume automatically.

In [ ]:
# Auto-detect repo root for JupyterLab / GCP / local execution

from pathlib import Path

import sys



def find_repo_root(start=None):

    start_path = Path(start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:

        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():

            return candidate

    return start_path



REPO_ROOT = find_repo_root()

DATA_ROOT = REPO_ROOT / 'data'

CKPT_ROOT = REPO_ROOT / 'checkpoints'

EXPS_ROOT = REPO_ROOT / 'experiments'

EXPS_ROOT.mkdir(exist_ok=True)

CKPT_ROOT.mkdir(exist_ok=True)



sys.path.insert(0, str(REPO_ROOT))



import torch

import torch.optim as optim

import numpy as np

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm



from src.data_loaders import get_shanghaitech_loaders, get_jhu_loaders

from src.models import CSRNet, AdaptiveCSRNet

from src.losses import CombinedDensityLoss

from src.training import DensityTrainer

from src.evaluation import evaluate_density

from src.utils import show_density_map, plot_training_curves, make_results_table



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

if torch.cuda.is_available():

    print(f'GPU: {torch.cuda.get_device_name(0)}')

    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 2: Training configuration ───────────────────────────────────────────
CFG = {
    # Data
    'sha_part'    : 'A',           # 'A' or 'B'
    'target_size' : (576, 768),    # (H, W) — set smaller (e.g. 288, 384) to save memory
    'batch_size'  : 4,             # reduce if OOM
    'workers'     : 2,

    # Training
    'epochs_baseline'  : 100,
    'epochs_adaptive'  : 150,
    'lr'               : 1e-5,
    'lr_adaptive'      : 5e-5,
    'weight_decay'     : 5e-4,
    'patience'         : 20,
}

# Reduce sizes for quick prototyping with CPU
if DEVICE == 'cpu':
    CFG['target_size'] = (288, 384)
    CFG['epochs_baseline'] = 3
    CFG['epochs_adaptive'] = 3
    print('CPU mode: reduced to 3 epochs for demo run')

print('Configuration:', CFG)

In [ ]:
# ── Cell 3: Load ShanghaiTech data ────────────────────────────────────────────
print(f'Loading ShanghaiTech Part {CFG["sha_part"]}...')
train_loader, test_loader = get_shanghaitech_loaders(
    data_root=str(DATA_ROOT),
    part=CFG['sha_part'],
    batch_size=CFG['batch_size'],
    target_size=CFG['target_size'],
    num_workers=CFG['workers'],
)

print(f'Train batches: {len(train_loader)}  ({len(train_loader.dataset)} images)')
print(f'Test  batches: {len(test_loader)}  ({len(test_loader.dataset)} images)')

# Peek at one batch
imgs, density, counts = next(iter(train_loader))
print(f'Batch shapes  → imgs: {tuple(imgs.shape)}, density: {tuple(density.shape)}, counts: {counts[:4].numpy()}')

In [ ]:
# ── Cell 4: Train CSRNet baseline ─────────────────────────────────────────────
print('=' * 60)
print('TRAINING CSRNet BASELINE')
print('=' * 60)

model_csrnet = CSRNet(load_weights=True).to(DEVICE)
print(f'CSRNet parameters: {sum(p.numel() for p in model_csrnet.parameters()):,}')

optimizer_csrnet = optim.Adam(
    model_csrnet.parameters(),
    lr=CFG['lr'], weight_decay=CFG['weight_decay']
)
scheduler_csrnet = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_csrnet, factor=0.5, patience=10, min_lr=1e-7, verbose=False
)

trainer_csrnet = DensityTrainer(
    model=model_csrnet,
    optimizer=optimizer_csrnet,
    scheduler=scheduler_csrnet,
    device=DEVICE,
    experiment_name=f'csrnet_sha{CFG["sha_part"]}',
    save_dir=str(CKPT_ROOT),
    log_dir=str(REPO_ROOT / 'runs'),
)

# Resume if checkpoint exists
trainer_csrnet.load_checkpoint('last.pt')

history_csrnet = trainer_csrnet.train(
    train_loader, test_loader,
    epochs=CFG['epochs_baseline'],
    patience=CFG['patience'],
    metric_key='mae',
)

In [ ]:
# ── Cell 5: Evaluate CSRNet ───────────────────────────────────────────────────
trainer_csrnet.load_checkpoint('best.pt')
csrnet_results = evaluate_density(model_csrnet, test_loader, DEVICE)

print('\nCSRNet Results on SHA-' + CFG['sha_part'] + ':')
print('-' * 40)
for k, v in csrnet_results.items():
    print(f'  {k:8s}: {v:.4f}')

In [ ]:
# ── Cell 6: Train AdaptiveCSRNet (our model) ──────────────────────────────────
print('=' * 60)
print('TRAINING AdaptiveCSRNet (OURS)')
print('=' * 60)

model_adaptive = AdaptiveCSRNet(load_weights=True, return_features=False).to(DEVICE)
n_params = sum(p.numel() for p in model_adaptive.parameters())
print(f'AdaptiveCSRNet parameters: {n_params:,}')

# Separate LR for pre-trained backbone vs new layers
backbone_params = list(model_adaptive.encoder.parameters())
new_params      = (list(model_adaptive.attention.parameters()) +
                   list(model_adaptive.backend.parameters()) +
                   list(model_adaptive.head.parameters()) +
                   list(model_adaptive.perspective_head.parameters()))

optimizer_adaptive = optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr_adaptive'] * 0.1},
    {'params': new_params,      'lr': CFG['lr_adaptive']},
], weight_decay=CFG['weight_decay'])

scheduler_adaptive = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_adaptive, T_max=CFG['epochs_adaptive'], eta_min=1e-7
)

loss_fn = CombinedDensityLoss(mse_weight=1.0, ssim_weight=0.5, tv_weight=1e-4)

trainer_adaptive = DensityTrainer(
    model=model_adaptive,
    optimizer=optimizer_adaptive,
    scheduler=scheduler_adaptive,
    loss_fn=loss_fn,
    device=DEVICE,
    experiment_name=f'adaptive_csrnet_sha{CFG["sha_part"]}',
    save_dir=str(CKPT_ROOT),
    log_dir=str(REPO_ROOT / 'runs'),
)

trainer_adaptive.load_checkpoint('last.pt')

history_adaptive = trainer_adaptive.train(
    train_loader, test_loader,
    epochs=CFG['epochs_adaptive'],
    patience=CFG['patience'],
    metric_key='mae',
)

In [ ]:
# ── Cell 7: Evaluate AdaptiveCSRNet ──────────────────────────────────────────
trainer_adaptive.load_checkpoint('best.pt')
adaptive_results = evaluate_density(model_adaptive, test_loader, DEVICE)

print('\nAdaptiveCSRNet Results on SHA-' + CFG['sha_part'] + ':')
print('-' * 40)
for k, v in adaptive_results.items():
    print(f'  {k:8s}: {v:.4f}')

In [ ]:
# ── Cell 8: Comparison table ──────────────────────────────────────────────────
all_results = {
    'CSRNet (baseline)' : csrnet_results,
    'AdaptiveCSRNet (ours)': adaptive_results,
}

table = make_results_table(all_results, columns=['mae', 'mse', 'psnr', 'ssim', 'game0', 'game1', 'game2', 'game3'])
print(f'\n=== ShanghaiTech Part {CFG["sha_part"]} Results ===')
print(table)

# Save to file
result_file = EXPS_ROOT / f'density_sha{CFG["sha_part"]}_results.md'
result_file.write_text(f'# SHA-{CFG["sha_part"]} Density Estimation Results\n\n' + table)
print(f'\nSaved to {result_file}')

In [ ]:
# ── Cell 9: Training curve comparison ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — SHA-{CFG["sha_part"]}', fontsize=13)

for label, history, color in [
    ('CSRNet',          history_csrnet,  'blue'),
    ('AdaptiveCSRNet',  history_adaptive,'red'),
]:
    train_mae = [e.get('mae', e.get('loss', 0)) for e in history['train']]
    val_mae   = [e.get('mae', 0) for e in history['val']]
    axes[0].plot(train_mae, color=color, linestyle='--', label=f'{label} (train)', alpha=0.7)
    axes[1].plot(val_mae,   color=color, label=f'{label} (val)')

for ax, title in zip(axes, ['Train Loss', 'Val MAE']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EXPS_ROOT / f'density_sha{CFG["sha_part"]}_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 10: Visual comparison on test samples ────────────────────────────────
import torchvision.transforms as T

model_csrnet.eval()
model_adaptive.eval()

# Get 3 test samples
test_iter = iter(test_loader)
n_show = min(3, len(test_loader))

denorm = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

for i in range(n_show):
    imgs, density_gt, counts_gt = next(test_iter)
    with torch.no_grad():
        pred_base = model_csrnet(imgs.to(DEVICE))
        pred_ours = model_adaptive(imgs.to(DEVICE))

    img_vis = denorm(imgs[0]).clamp(0, 1)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'Test Sample {i+1}  |  GT Count: {counts_gt[0].item():.0f}', fontsize=12)

    axes[0].imshow(img_vis.permute(1,2,0).numpy())
    axes[0].set_title('Input'); axes[0].axis('off')

    axes[1].imshow(density_gt[0,0].numpy(), cmap='jet')
    axes[1].set_title(f'GT Density (sum={density_gt[0,0].sum():.0f})')
    axes[1].axis('off')

    p_base = pred_base[0,0].cpu().numpy()
    axes[2].imshow(p_base, cmap='jet')
    axes[2].set_title(f'CSRNet (count={p_base.sum():.0f})')
    axes[2].axis('off')

    p_ours = pred_ours[0,0].cpu().numpy()
    axes[3].imshow(p_ours, cmap='jet')
    axes[3].set_title(f'Ours (count={p_ours.sum():.0f})')
    axes[3].axis('off')

    plt.tight_layout()
    plt.savefig(EXPS_ROOT / f'density_sample_{i+1}.png', dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# ── Cell 11: Also train on JHU-Crowd++ (if available) ─────────────────────────
jhu_train_dir = DATA_ROOT / 'jhu_crowd_v2.0' / 'train'

if jhu_train_dir.exists():
    print('JHU-Crowd++ dataset found. Training AdaptiveCSRNet on JHU...')
    
    jhu_train, jhu_val, jhu_test = get_jhu_loaders(
        data_root=str(DATA_ROOT),
        batch_size=CFG['batch_size'],
        target_size=CFG['target_size'],
        num_workers=CFG['workers'],
    )
    print(f'JHU Train: {len(jhu_train.dataset)} | Val: {len(jhu_val.dataset)} | Test: {len(jhu_test.dataset)}')

    model_jhu = AdaptiveCSRNet(load_weights=True).to(DEVICE)
    opt_jhu = optim.AdamW(model_jhu.parameters(), lr=CFG['lr_adaptive'], weight_decay=CFG['weight_decay'])
    sched_jhu = optim.lr_scheduler.CosineAnnealingLR(opt_jhu, T_max=CFG['epochs_adaptive'], eta_min=1e-7)

    trainer_jhu = DensityTrainer(
        model=model_jhu, optimizer=opt_jhu, scheduler=sched_jhu,
        loss_fn=CombinedDensityLoss(), device=DEVICE,
        experiment_name='adaptive_csrnet_jhu',
        save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
    )
    trainer_jhu.load_checkpoint('last.pt')
    history_jhu = trainer_jhu.train(
        jhu_train, jhu_val,
        epochs=CFG['epochs_adaptive'], patience=CFG['patience']
    )

    trainer_jhu.load_checkpoint('best.pt')
    jhu_results = evaluate_density(model_jhu, jhu_test, DEVICE)
    print('\nJHU-Crowd++ Results:')
    for k, v in jhu_results.items():
        print(f'  {k:8s}: {v:.4f}')
else:
    print('JHU-Crowd++ not found at expected path, skipping.')

---
## Density Estimation Complete!

Results and visualisations saved to `experiments/`. Continue with **02_forecasting.ipynb**.